Import libraries

In [9]:
import numpy as np
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import time

Path to QA-Prolog compatible ontology

In [10]:
file = '../knowledge_bases/Animal_QA_Prolog_compatible.pl'

Defining queries

In [ ]:
query_list = []
query_list.append("class(animal).")
query_list.append("class(X).")
query_list.append("subClass(seal, X).")
query_list.append("isA(reiny, X).")
query_list.append("isA(sardy, X).")
query_list.append("isA(Y, X).")
query_list.append("error(X, Y, Z).")
query_list.append("hasProperty(I1, ancestor, reiny_c).")

Defining commands to run the pipeline and to generate a matrix representation of the QUBO problem

In [ ]:
command_QAP = 'QA-Prolog --qmasm-args="--stop" --work-dir="' + str(Path.cwd()) + '/work"'
command_qmasm = 'qmasm --format="numpy" -o="out.npz" --pin="Query.Valid := true" --solver="tabu" ' + str(Path.cwd()) + '/work/Animal_QA_Prolog_compatible.qmasm'

Function to run the pipeline, generate the matrix representing the QUBO problem and retreive its dimension

In [13]:
def experiment(query):
    command = command_QAP + ' --query="' + query + '" ' + file
    _ = subprocess.run(command, shell=True, capture_output=True, text=True)
    _ = subprocess.run(command_qmasm, shell=True)
    data = np.load("out.npz")
    return len(data["syms"])


Run `experiment` function for each query and accumulate the results (taking execution time for each query)

In [ ]:
n_qubits = []
for i,query in enumerate(query_list):
    print('query ', i+1, '/', len(query_list), ': ', query, '...', end="", sep="")
    start = time.time()
    n_qubits.append(experiment(query))
    stop = time.time()
    print('ok ', '(', int(stop-start) ,' sec)', sep="", )


query 1/8: class(animal)....ok (3 sec)
query 2/8: class(X)....ok (2 sec)
query 3/8: subClass(seal, X)....ok (7 sec)
query 4/8: isA(reiny, X)...ok (21 sec)
query 5/8: isA(sardy, X)...ok (20 sec)
query 6/8: isA(Y, X)...ok (20 sec)
query 7/8: error(X, Y, Z)...ok (54 sec)
query 8/8: hasProperty(I1, ancestor, reiny_c)....

Print the results and save them as a *tab separeted value* file

In [1]:
with open('raw_data.tsv', 'w') as out:
    out.write('query\tn_qubits\n')
    for i in range(len(query_list)):
        print(query_list[i], n_qubits[i])
        out.write(query[i]+'\t'+str(n_qubits[i])+'\n')

NameError: name 'query_list' is not defined

Plot the data

In [ ]:
fig, ax = plt.subplots()
ax.plot(n_qubits)
plt.show()